# Week 9: Panel Data Fundamentals

**QM 2023 — Statistics II / Data Analytics**
**Spring 2026 | University of Tulsa**

---

## What You Will Learn

By the end of this demo, you will be able to:

1. **Define** panel data and explain how it differs from cross-sectional and time-series data
2. **Reshape** data between wide and long formats using pandas
3. **Calculate** panel dimensions: N (entities), T (time periods), and total observations
4. **Distinguish** balanced vs. unbalanced panels
5. **Decompose** variation into "between" (across firms) and "within" (over time) components
6. **Explain** why panel structure matters for regression — and why pooled OLS can mislead

---

### The Big Picture

Until now, we have worked with **cross-sectional data** (many firms at one point in time) or **time-series data** (one firm over many time periods). **Panel data** combines both — we observe the *same* entities repeatedly over time. This simple structural change unlocks powerful techniques for causal inference that we will explore in Weeks 10-11.

---

## Setup: Imports and Configuration

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

# Plot style settings for consistent, beginner-friendly visuals
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['figure.figsize'] = (10, 6)

print("Imports successful!")

---

## Section 1: What is Panel Data?

There are three fundamental data structures in empirical research:

| Structure | What it tracks | Example |
|:---|:---|:---|
| **Cross-section** | Many entities, one time period | Revenue of 500 firms in 2024 |
| **Time series** | One entity, many time periods | Apple's quarterly revenue 2015-2024 |
| **Panel data** | Many entities, many time periods | Revenue of 500 firms from 2015-2024 |

Panel data is the most informative structure because it lets us:
- **Control for unobserved differences** between entities (e.g., management quality)
- **Track how entities change** over time in response to events or policies
- **Increase sample size** dramatically (N entities x T periods = N*T observations)

Think of it this way: a cross-section is a **photograph**, a time series is a **video of one person**, and a panel is a **video of an entire classroom**.

---

## Section 2: Create a Simulated Panel Dataset

We will build a small, transparent panel dataset from scratch: **10 firms observed over 5 years**. This gives us 50 total observations — small enough to inspect every row, large enough to see real patterns.

Each firm has:
- **Revenue** (millions): varies by firm and over time
- **Employees** (count): grows over time, differs by firm
- **RD_Spending** (R&D, millions): research investment

We also give each firm an unobserved "management quality" score. This hidden factor affects revenue but is NOT in our data — it represents the kind of **omitted variable** that makes panel methods so valuable.

In [ ]:
np.random.seed(42)

# Panel dimensions
n_firms = 10          # N = number of entities (firms)
n_years = 5           # T = number of time periods (years)
firm_names = ['AlphaREIT', 'BetaCorp', 'GammaTrust', 'DeltaFund', 'EpsilonProp',
              'ZetaHoldings', 'EtaCapital', 'ThetaGroup', 'IotaInvest', 'KappaAssets']
years = [2020, 2021, 2022, 2023, 2024]

# Unobserved firm-specific "management quality" — this is the HIDDEN factor
# Some firms are just better-managed, which boosts revenue permanently
# Critically: better-managed firms ALSO tend to have more employees (correlation!)
management_quality = np.array([8, 2, 6, -2, 4, 10, 1, 7, -1, 5])  # Fixed for each firm

print(f"Panel dimensions: N = {n_firms} firms, T = {n_years} years")
print(f"Total observations: N x T = {n_firms * n_years}")
print(f"\nFirm names: {firm_names}")
print(f"Years: {years}")
print(f"\nManagement quality (unobserved): {dict(zip(firm_names, management_quality))}")

In [ ]:
# Build the panel dataset row by row — one row per firm-year pair
rows = []
for i in range(n_firms):
    for t in range(n_years):
        firm_name = firm_names[i]
        year = years[t]

        # Employees: base depends on firm, grows ~5% per year, plus noise
        base_employees = 100 + i * 30                          # Firms differ in size
        employees = int(base_employees * (1.05 ** t) + np.random.normal(0, 10))

        # R&D spending: base depends on firm, small random changes each year
        base_rd = 2 + i * 0.5                                 # Bigger firms spend more on R&D
        rd_spending = round(base_rd + t * 0.3 + np.random.normal(0, 0.5), 1)

        # Revenue: TRUE relationship is Revenue = 0.5 * Employees + 3 * RD + management_quality + noise
        # The management_quality term is UNOBSERVED — this is what makes panel methods necessary
        revenue = round(0.5 * employees + 3 * rd_spending + management_quality[i] * 5
                        + np.random.normal(0, 8), 1)

        rows.append({
            'Firm': firm_name,
            'Year': year,
            'Revenue': revenue,
            'Employees': employees,
            'RD_Spending': rd_spending
        })

panel_df = pd.DataFrame(rows)

print(f"Dataset shape: {panel_df.shape[0]} rows x {panel_df.shape[1]} columns")
print(f"\nFirst 15 rows (first 3 firms, all years):")
display(panel_df.head(15))

### Interpretation Checkpoint: Inspect the Raw Data

Look at the output above. Notice the structure:
- **Each firm appears 5 times** (once for each year)
- **Rows are sorted** by firm, then by year within each firm
- This is **long format** — the most common format for panel analysis in Python

**Question:** How many rows would you expect for 10 firms over 5 years? Count them!

In [ ]:
# Quick summary statistics for the full panel
print("Descriptive Statistics for Our Panel:")
print("=" * 50)
display(panel_df.describe().round(1))

print(f"\nUnique firms: {panel_df['Firm'].nunique()}")
print(f"Unique years: {panel_df['Year'].nunique()}")
print(f"Year range: {panel_df['Year'].min()} to {panel_df['Year'].max()}")

---

## Section 3: Wide vs. Long Format

This is one of the most important practical concepts in panel data. The **same data** can be stored in two very different shapes:

- **Long format** (what we just built): One row per firm-year. This is what Python regression functions expect.
- **Wide format**: One row per firm, with separate columns for each year. This is what you often see in spreadsheets or financial databases.

Let's see both formats side by side for **Revenue**.

In [ ]:
# Convert long format to wide format using pivot
# Each firm becomes ONE row, with Revenue columns for each year
revenue_wide = panel_df.pivot(index='Firm', columns='Year', values='Revenue')

# Rename columns to be clear
revenue_wide.columns = [f'Revenue_{yr}' for yr in revenue_wide.columns]

print("WIDE FORMAT — One row per firm, one column per year:")
print("=" * 70)
display(revenue_wide)

In [ ]:
# Compare with long format — same data, different shape
revenue_long = panel_df[['Firm', 'Year', 'Revenue']].copy()

print("LONG FORMAT — One row per firm-year pair:")
print("=" * 70)
display(revenue_long.head(15))

print(f"\nWide format shape: {revenue_wide.shape[0]} rows x {revenue_wide.shape[1]} columns")
print(f"Long format shape:  {revenue_long.shape[0]} rows x {revenue_long.shape[1]} columns")
print(f"\nSame data, different shape! Wide has fewer rows but more columns.")

In [ ]:
# Converting BACK from wide to long using melt
# This is one of the most common operations in real-world data wrangling
revenue_back_to_long = revenue_wide.reset_index().melt(
    id_vars='Firm',                    # Column(s) that identify each entity
    var_name='Year_Label',             # Name for the new "variable" column
    value_name='Revenue'               # Name for the new "value" column
)

print("Wide -> Long using pd.melt():")
print("=" * 50)
display(revenue_back_to_long.head(10))

print("\nKey pandas functions for reshaping:")
print("  Long -> Wide:  df.pivot(index='Firm', columns='Year', values='Revenue')")
print("  Wide -> Long:  df.melt(id_vars='Firm', var_name='Year', value_name='Revenue')")

### Interpretation Checkpoint: Wide vs. Long

- **Wide format** is easier to read in a spreadsheet — you can scan across a row to see one firm's history
- **Long format** is what Python regression tools require — each observation is one row
- In practice, you will receive data in one format and need to convert to the other
- **`pivot()`** goes long-to-wide; **`melt()`** goes wide-to-long. Memorize these two!

---

## Section 4: Panel Dimensions — Balanced vs. Unbalanced

A **balanced panel** has every entity observed in every time period (no gaps). An **unbalanced panel** has missing observations — some firms may enter or exit the data partway through.

Our simulated data is balanced. Let's verify, and then see what an unbalanced panel looks like.

In [ ]:
# Count how many years each firm appears in
observations_per_firm = panel_df.groupby('Firm')['Year'].count()

print("Observations per firm:")
print("=" * 40)
print(observations_per_firm.to_string())
print(f"\nAll firms have {observations_per_firm.unique()[0]} observations — this is a BALANCED panel.")
print(f"\nN = {panel_df['Firm'].nunique()} firms")
print(f"T = {panel_df['Year'].nunique()} years")
print(f"N x T = {panel_df['Firm'].nunique()} x {panel_df['Year'].nunique()} = {len(panel_df)} total observations")

In [ ]:
# Create an UNBALANCED panel by dropping some observations
# In real data, firms go bankrupt, get acquired, or simply stop reporting
unbalanced_df = panel_df.copy()

# Drop BetaCorp's 2020-2021 data (firm entered market late)
unbalanced_df = unbalanced_df[~((unbalanced_df['Firm'] == 'BetaCorp') & 
                                 (unbalanced_df['Year'].isin([2020, 2021])))]

# Drop KappaAssets' 2024 data (firm was acquired)
unbalanced_df = unbalanced_df[~((unbalanced_df['Firm'] == 'KappaAssets') & 
                                 (unbalanced_df['Year'] == 2024))]

# Count observations per firm in unbalanced panel
obs_per_firm_unbalanced = unbalanced_df.groupby('Firm')['Year'].count()

print("UNBALANCED PANEL — observations per firm:")
print("=" * 50)
print(obs_per_firm_unbalanced.to_string())
print(f"\nTotal observations: {len(unbalanced_df)} (was {len(panel_df)} in balanced panel)")
print(f"Missing: {len(panel_df) - len(unbalanced_df)} observations")
print("\nBetaCorp entered late (missing 2020-2021), KappaAssets was acquired (missing 2024)")

In [ ]:
# Visualize panel coverage with a heatmap — a great way to spot gaps
# Create a firm-year indicator matrix: 1 = observed, 0 = missing
coverage_matrix = unbalanced_df.pivot_table(
    index='Firm', columns='Year', values='Revenue', aggfunc='count'
).fillna(0)

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(coverage_matrix, annot=True, fmt='.0f', cmap='YlGn',
            cbar_kws={'label': '1 = Observed, 0 = Missing'},
            linewidths=0.5, ax=ax)
ax.set_title('Panel Coverage Heatmap — Unbalanced Panel\n(Gaps show missing firm-year observations)')
ax.set_ylabel('Firm')
ax.set_xlabel('Year')
plt.tight_layout()
plt.show()

### Interpretation Checkpoint: Balanced vs. Unbalanced

- The heatmap makes gaps instantly visible — BetaCorp has no 2020-2021 data, KappaAssets has no 2024 data
- **Balanced panels** are convenient but rare in practice. Real-world data almost always has gaps
- Unbalanced panels are NOT errors — firms enter and exit markets, and that is perfectly realistic
- Most panel methods (including fixed effects) handle unbalanced panels just fine

**For the rest of this demo, we will use our balanced panel** (all 10 firms, all 5 years) to keep things clean.

---

## Section 5: Within vs. Between Variation

This is the **single most important concept** in this demo. Understanding this unlocks everything in Weeks 10-11.

When we look at a variable like Revenue across our panel, the total variation comes from two sources:

1. **Between variation**: How much do firms differ from *each other* on average? (AlphaREIT is a big firm, DeltaFund is small)
2. **Within variation**: How much does *each firm* change over time? (AlphaREIT's revenue goes up and down year to year)

Think of it this way:
- **Between** = the differences you see in a **group photo** (some people are tall, some are short)
- **Within** = the changes you see in a **time-lapse** of one person (they grow taller each year)

In [ ]:
# Step 1: Calculate the MEAN revenue for each firm (across all years)
mean_revenue_by_firm = panel_df.groupby('Firm')['Revenue'].mean()

print("Average Revenue by Firm (the 'between' component):")
print("=" * 50)
print(mean_revenue_by_firm.round(1).to_string())

print(f"\nOverall mean revenue: {panel_df['Revenue'].mean():.1f}")
print(f"Highest average: {mean_revenue_by_firm.idxmax()} ({mean_revenue_by_firm.max():.1f})")
print(f"Lowest average:  {mean_revenue_by_firm.idxmin()} ({mean_revenue_by_firm.min():.1f})")
print(f"\nThe SPREAD of these firm averages is the 'between' variation.")

In [ ]:
# Visualize the between variation — bar chart of firm average revenues
fig, ax = plt.subplots(figsize=(10, 5))

# Sort firms by average revenue for cleaner visualization
sorted_means = mean_revenue_by_firm.sort_values()
colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(sorted_means)))

ax.barh(sorted_means.index, sorted_means.values, color=colors, edgecolor='white')
ax.axvline(panel_df['Revenue'].mean(), color='red', linestyle='--', linewidth=2,
           label=f'Overall Mean = {panel_df["Revenue"].mean():.1f}')
ax.set_xlabel('Average Revenue (millions)')
ax.set_title('BETWEEN Variation — How Firms Differ From Each Other\n(Each bar = one firm\'s average revenue across all years)')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

In [ ]:
# Step 2: Calculate the WITHIN variation — how each firm deviates from its own mean
# For each observation: within_deviation = Revenue - that firm's average revenue
panel_df['firm_mean_revenue'] = panel_df.groupby('Firm')['Revenue'].transform('mean')
panel_df['within_revenue'] = panel_df['Revenue'] - panel_df['firm_mean_revenue']

print("Within-firm deviations (Revenue minus firm mean):")
print("=" * 70)
# Show for first 3 firms
for firm in firm_names[:3]:
    firm_data = panel_df[panel_df['Firm'] == firm][['Firm', 'Year', 'Revenue', 'firm_mean_revenue', 'within_revenue']]
    print(f"\n{firm} (mean revenue = {firm_data['firm_mean_revenue'].iloc[0]:.1f}):")
    display(firm_data.round(1))

In [ ]:
# Spaghetti plot — each firm's revenue trajectory over time
# This is the CLASSIC panel data visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left panel: Raw revenue trajectories (shows BOTH between and within variation)
ax1 = axes[0]
for firm in firm_names:
    firm_data = panel_df[panel_df['Firm'] == firm]
    ax1.plot(firm_data['Year'], firm_data['Revenue'], marker='o', linewidth=2, 
             label=firm, alpha=0.8)
ax1.set_xlabel('Year')
ax1.set_ylabel('Revenue (millions)')
ax1.set_title('Raw Revenue Trajectories\n(Shows BOTH between and within variation)')
ax1.legend(fontsize=7, loc='upper left', ncol=2)
ax1.grid(True, alpha=0.3)

# Right panel: De-meaned revenue (within variation ONLY — firm averages removed)
ax2 = axes[1]
for firm in firm_names:
    firm_data = panel_df[panel_df['Firm'] == firm]
    ax2.plot(firm_data['Year'], firm_data['within_revenue'], marker='o', linewidth=2,
             label=firm, alpha=0.8)
ax2.axhline(0, color='black', linestyle='-', linewidth=1)
ax2.set_xlabel('Year')
ax2.set_ylabel('Revenue Deviation from Firm Mean')
ax2.set_title('De-meaned Revenue (Within Variation ONLY)\n(Each firm centered at zero — between variation removed)')
ax2.legend(fontsize=7, loc='upper left', ncol=2)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Interpretation Checkpoint: What Do You Notice?

Compare the two plots above:

- **Left plot** (raw data): The lines are spread far apart vertically. ZetaHoldings and AlphaREIT are at the top, DeltaFund is at the bottom. This vertical spread is **between variation** — firms are just different sizes.
- **Right plot** (de-meaned): All firms are now centered around zero. The remaining wiggle is **within variation** — how each firm changes relative to its own average.

**Key insight:** The right plot removes the permanent differences between firms. What remains is each firm's year-to-year fluctuation. Fixed Effects models (Week 10) work exactly this way — they "de-mean" the data to focus on within variation only.

In [ ]:
# Formal variance decomposition for Revenue
# Total variance = Between variance + Within variance

total_variance_revenue = panel_df['Revenue'].var()

# Between variance: variance of the firm means
between_variance_revenue = mean_revenue_by_firm.var()

# Within variance: average of within-firm variances
within_variance_revenue = panel_df.groupby('Firm')['Revenue'].var().mean()

print("VARIANCE DECOMPOSITION — Revenue")
print("=" * 50)
print(f"Total variance:   {total_variance_revenue:>10.1f}")
print(f"Between variance: {between_variance_revenue:>10.1f}  ({100 * between_variance_revenue / total_variance_revenue:.1f}%)")
print(f"Within variance:  {within_variance_revenue:>10.1f}  ({100 * within_variance_revenue / total_variance_revenue:.1f}%)")
print(f"\nInterpretation:")
print(f"  {100 * between_variance_revenue / total_variance_revenue:.0f}% of revenue variation is BETWEEN firms (firms differ from each other)")
print(f"  {100 * within_variance_revenue / total_variance_revenue:.0f}% of revenue variation is WITHIN firms (firms change over time)")

In [ ]:
# Do the same decomposition for all variables — build an intuitive comparison table
variables_to_decompose = ['Revenue', 'Employees', 'RD_Spending']

decomposition_rows = []
for var in variables_to_decompose:
    total_var = panel_df[var].var()
    between_var = panel_df.groupby('Firm')[var].mean().var()
    within_var = panel_df.groupby('Firm')[var].var().mean()

    decomposition_rows.append({
        'Variable': var,
        'Total Variance': round(total_var, 1),
        'Between Variance': round(between_var, 1),
        'Within Variance': round(within_var, 1),
        'Between %': f"{100 * between_var / total_var:.0f}%",
        'Within %': f"{100 * within_var / total_var:.0f}%"
    })

decomposition_table = pd.DataFrame(decomposition_rows)
print("Variance Decomposition — All Variables:")
print("=" * 70)
display(decomposition_table)

### Interpretation Checkpoint: Variance Decomposition

Look at the table above:
- **Employees** likely has very high between variation — firms differ a lot in size, but each firm's headcount is fairly stable year to year
- **R&D Spending** may show more within variation — R&D budgets can shift year to year within the same firm

**Why does this matter?** Fixed Effects models (Week 10) can only use **within variation** to estimate coefficients. If a variable has almost no within variation (it barely changes over time for each firm), fixed effects will struggle to estimate its effect. This is a real limitation to keep in mind!

---

## Section 6: Why Panel Structure Matters for Regression

Now we come to the payoff. Let's see what happens when we run a standard pooled OLS regression on panel data — **ignoring** the panel structure entirely. Then we will compare it to what the true relationship is.

Recall our data-generating process:

$$\text{Revenue} = 0.5 \times \text{Employees} + 3.0 \times \text{RD\_Spending} + 5 \times \text{Management\_Quality} + \text{noise}$$

The problem: **Management Quality is unobserved** (it is not in our dataset). It is a firm-specific trait that affects Revenue but is also correlated with firm size (bigger, better-managed firms tend to have more employees). This is **omitted variable bias** — exactly the problem that panel methods solve.

In [ ]:
# Pooled OLS — treats all 50 observations as independent, ignores firm structure
# This is what we learned in Weeks 4-6, applied naively to panel data

y_revenue = panel_df['Revenue']                           # Outcome: Revenue
X_predictors = panel_df[['Employees', 'RD_Spending']]     # Predictors (Management Quality is MISSING)
X_design = sm.add_constant(X_predictors)                  # Add intercept

pooled_ols_model = sm.OLS(y_revenue, X_design).fit()

print("POOLED OLS — Ignoring Panel Structure")
print("=" * 60)
print(f"\nEstimated equation:")
print(f"  Revenue = {pooled_ols_model.params['const']:.1f} "
      f"+ {pooled_ols_model.params['Employees']:.3f} * Employees "
      f"+ {pooled_ols_model.params['RD_Spending']:.2f} * RD_Spending")
print(f"\n  R-squared: {pooled_ols_model.rsquared:.4f}")
print(f"\nTRUE coefficients:  Employees = 0.500,  RD_Spending = 3.00")
print(f"Pooled OLS:         Employees = {pooled_ols_model.params['Employees']:.3f},  "
      f"RD_Spending = {pooled_ols_model.params['RD_Spending']:.2f}")

# Check if biased
emp_bias = pooled_ols_model.params['Employees'] - 0.5
rd_bias = pooled_ols_model.params['RD_Spending'] - 3.0
print(f"\nBias on Employees:   {emp_bias:+.3f}  {'(BIASED!)' if abs(emp_bias) > 0.05 else '(close)'}")
print(f"Bias on RD_Spending: {rd_bias:+.2f}  {'(BIASED!)' if abs(rd_bias) > 0.5 else '(close)'}")

### What went wrong?

The Employees coefficient is **biased** because:
1. Management Quality affects Revenue (better-managed firms earn more)
2. Management Quality is correlated with Employees (better-managed firms are larger)
3. Since Management Quality is **omitted**, its effect gets incorrectly absorbed into the Employees coefficient

This is classic **omitted variable bias**. The pooled OLS coefficient on Employees is too large because it captures both the direct effect of employees AND the indirect effect of better management.

**The fix?** Fixed Effects (Week 10). By including a separate intercept for each firm, we absorb all time-invariant firm characteristics — including unobserved management quality.

In [ ]:
# Quick preview: what happens if we de-mean the data (remove firm averages)?
# This is exactly what Fixed Effects does — we will formalize it in Week 10

# De-mean each variable by firm
panel_df['Employees_demeaned'] = panel_df['Employees'] - panel_df.groupby('Firm')['Employees'].transform('mean')
panel_df['RD_demeaned'] = panel_df['RD_Spending'] - panel_df.groupby('Firm')['RD_Spending'].transform('mean')
panel_df['Revenue_demeaned'] = panel_df['Revenue'] - panel_df.groupby('Firm')['Revenue'].transform('mean')

# Run OLS on de-meaned data (this IS the Fixed Effects estimator!)
y_demeaned = panel_df['Revenue_demeaned']
X_demeaned = panel_df[['Employees_demeaned', 'RD_demeaned']]
# Note: no constant needed — de-meaning removes it

fe_preview_model = sm.OLS(y_demeaned, X_demeaned).fit()

print("PREVIEW: De-meaned OLS (= Fixed Effects)")
print("=" * 60)
print(f"\n  Employees coefficient: {fe_preview_model.params['Employees_demeaned']:.3f}  (true: 0.500)")
print(f"  RD_Spending coefficient: {fe_preview_model.params['RD_demeaned']:.2f}  (true: 3.00)")
print(f"\nCompare to Pooled OLS:")
print(f"  Employees: Pooled = {pooled_ols_model.params['Employees']:.3f}, De-meaned = {fe_preview_model.params['Employees_demeaned']:.3f}")
print(f"  RD_Spending: Pooled = {pooled_ols_model.params['RD_Spending']:.2f}, De-meaned = {fe_preview_model.params['RD_demeaned']:.2f}")
print(f"\nDe-meaning removes firm-level heterogeneity and corrects the bias!")

In [ ]:
# Visual comparison: Pooled OLS vs De-meaned (Fixed Effects preview)
fig, ax = plt.subplots(figsize=(9, 5))

variables = ['Employees', 'RD_Spending']
true_values = [0.5, 3.0]
pooled_values = [pooled_ols_model.params['Employees'], pooled_ols_model.params['RD_Spending']]
fe_values = [fe_preview_model.params['Employees_demeaned'], fe_preview_model.params['RD_demeaned']]

x_pos = np.arange(len(variables))
width = 0.25

bars_true = ax.bar(x_pos - width, true_values, width, label='True Value', color='green', alpha=0.8)
bars_pooled = ax.bar(x_pos, pooled_values, width, label='Pooled OLS (biased)', color='red', alpha=0.8)
bars_fe = ax.bar(x_pos + width, fe_values, width, label='De-meaned / FE (corrected)', color='blue', alpha=0.8)

ax.set_xticks(x_pos)
ax.set_xticklabels(variables, fontsize=12)
ax.set_ylabel('Coefficient Estimate', fontsize=11)
ax.set_title('Coefficient Comparison: True vs Pooled OLS vs Fixed Effects\n(Pooled OLS is biased when unobserved firm effects exist)',
             fontsize=12)
ax.legend(fontsize=10)
ax.axhline(0, color='black', linewidth=0.5)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

### Interpretation Checkpoint: The Power of Panel Data

Look at the bar chart above:
- **Green bars** = the true coefficients we used to generate the data
- **Red bars** = what pooled OLS estimates (biased — especially for Employees)
- **Blue bars** = what the de-meaned estimator recovers (much closer to truth!)

The blue bars are close to the green bars because de-meaning removes the unobserved firm effects. This is exactly what Fixed Effects regression does, and we will learn the formal version in **Week 10**.

**Bottom line:** Panel data + Fixed Effects lets us control for things we cannot even measure, as long as those things are constant over time for each entity.

In [ ]:
# One more visualization: scatter plot showing the pooled vs within-firm relationship
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Pooled scatter (Employees vs Revenue) — mixes between and within variation
ax1 = axes[0]
colors_map = plt.cm.tab10(np.linspace(0, 1, n_firms))
for i, firm in enumerate(firm_names):
    firm_data = panel_df[panel_df['Firm'] == firm]
    ax1.scatter(firm_data['Employees'], firm_data['Revenue'], 
                color=colors_map[i], s=60, label=firm, alpha=0.8, edgecolors='k', linewidth=0.3)

# Add pooled regression line
x_range_pooled = np.linspace(panel_df['Employees'].min(), panel_df['Employees'].max(), 100)
y_pred_pooled = pooled_ols_model.params['const'] + pooled_ols_model.params['Employees'] * x_range_pooled
ax1.plot(x_range_pooled, y_pred_pooled, 'r-', linewidth=2.5, label='Pooled OLS line')
ax1.set_xlabel('Employees')
ax1.set_ylabel('Revenue (millions)')
ax1.set_title('Pooled OLS: One Line for All Firms\n(Mixes between + within variation)')
ax1.legend(fontsize=6, ncol=2, loc='upper left')
ax1.grid(True, alpha=0.3)

# Right: Same scatter but with separate lines per firm (within-firm relationships)
ax2 = axes[1]
for i, firm in enumerate(firm_names):
    firm_data = panel_df[panel_df['Firm'] == firm]
    ax2.scatter(firm_data['Employees'], firm_data['Revenue'],
                color=colors_map[i], s=60, alpha=0.8, edgecolors='k', linewidth=0.3)
    # Fit a mini regression line for each firm
    z = np.polyfit(firm_data['Employees'], firm_data['Revenue'], 1)
    p = np.poly1d(z)
    x_firm = np.linspace(firm_data['Employees'].min(), firm_data['Employees'].max(), 50)
    ax2.plot(x_firm, p(x_firm), color=colors_map[i], linewidth=1.5, alpha=0.7)

ax2.set_xlabel('Employees')
ax2.set_ylabel('Revenue (millions)')
ax2.set_title('Within-Firm Relationships: Separate Line Per Firm\n(Each firm has its own intercept = Fixed Effects idea)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Interpretation Checkpoint: Pooled vs. Within-Firm

These two scatter plots tell the whole story:

- **Left plot**: One regression line for everyone. The steep slope is driven by the fact that **big firms** (many employees) also have **high management quality** (an omitted variable). The line conflates "more employees cause more revenue" with "better-managed firms happen to be bigger."

- **Right plot**: Each firm gets its own line (its own intercept). Now the slope within each firm reflects only the **within-firm** relationship: when THIS firm hires more people, how much does ITS revenue change? This is the Fixed Effects idea.

**The right plot is the intuition behind Fixed Effects regression.** Each firm-specific intercept absorbs all the permanent, unobserved differences between firms.

---

## Summary: Key Takeaways

### 1. Panel Data = Entities x Time
Panel data tracks the **same** entities (firms, people, countries) over **multiple** time periods. This is more informative than a cross-section or a time series alone.

### 2. Wide vs. Long Format
- **Wide**: One row per entity, columns for each time period. Good for spreadsheets.
- **Long**: One row per entity-time pair. **Required for regression in Python.**
- Use `pivot()` (long-to-wide) and `melt()` (wide-to-long) to convert.

### 3. Balanced vs. Unbalanced
- **Balanced**: Every entity observed in every period (no gaps).
- **Unbalanced**: Some entities have missing periods. This is normal in real data.

### 4. Within vs. Between Variation (THE Key Concept)
- **Between variation**: How entities differ from each other on average.
- **Within variation**: How each entity changes over time relative to its own mean.
- Fixed Effects models use **only within variation** to estimate coefficients.

### 5. Why It Matters for Regression
- Pooled OLS ignores panel structure and can suffer from **omitted variable bias**.
- De-meaning the data (subtracting entity means) removes time-invariant confounders.
- This is exactly what Fixed Effects regression does — coming in **Week 10**.

---

### Coming Up Next
- **Assignment 09**: You will reshape real REIT data into panel format and decompose within/between variation.
- **Week 10**: Fixed Effects regression — the formal estimation method that exploits within variation.
- **Week 11**: Difference-in-Differences — using panel data to evaluate the causal effect of policy changes.

In [ ]:
# Final summary printout
print("=" * 60)
print("WEEK 9 PANEL DATA FUNDAMENTALS — SUMMARY")
print("=" * 60)

print(f"\nPanel structure:")
print(f"  N = {n_firms} firms, T = {n_years} years, N x T = {n_firms * n_years} observations")

print(f"\nVariance decomposition (Revenue):")
print(f"  Between (across firms):  {100 * between_variance_revenue / total_variance_revenue:.0f}%")
print(f"  Within (over time):      {100 * within_variance_revenue / total_variance_revenue:.0f}%")

print(f"\nPooled OLS vs Fixed Effects (Employees coefficient):")
print(f"  True value:    0.500")
print(f"  Pooled OLS:    {pooled_ols_model.params['Employees']:.3f}  (biased by omitted management quality)")
print(f"  De-meaned/FE:  {fe_preview_model.params['Employees_demeaned']:.3f}  (bias corrected!)")

print(f"\nKey functions to remember:")
print(f"  pd.pivot()  — long to wide")
print(f"  pd.melt()   — wide to long")
print(f"  .groupby().transform('mean')  — compute group means for de-meaning")

print(f"\nNext week: Fixed Effects regression with linearmodels.PanelOLS")